# Feature Engineering

In [1]:
import os
from collections import defaultdict, deque

import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import drive

drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/ML Assignment"
RAW_SPLIT_DIR = os.path.join(BASE_DIR, "splitted_dataset")
PREPROCESSED_DIR = os.path.join(BASE_DIR, "preprocessed_dataset")
FEATURE_DIR = os.path.join(BASE_DIR, "engineered_dataset")

os.makedirs(FEATURE_DIR, exist_ok=True)

print("Environment setup completed.")

Mounted at /content/drive
Environment setup completed.


Load splitted dataset

In [2]:
train_raw = pd.read_csv(
    os.path.join(RAW_SPLIT_DIR, "train.csv")
)

val_raw = pd.read_csv(
    os.path.join(RAW_SPLIT_DIR, "validation.csv")
)

test_raw = pd.read_csv(
    os.path.join(RAW_SPLIT_DIR, "test.csv")
)

print("Training raw:", train_raw.shape)
print("Validation raw:", val_raw.shape)
print("Test raw:", test_raw.shape)

Training raw: (18183, 121)
Validation raw: (3873, 121)
Test raw: (3923, 121)


Load preprocessed datasets

In [3]:
X_train = pd.read_csv(os.path.join(PREPROCESSED_DIR, "X_train_preprocessed.csv"))
X_val = pd.read_csv(os.path.join(PREPROCESSED_DIR, "X_validation_preprocessed.csv"))
X_test = pd.read_csv(os.path.join(PREPROCESSED_DIR, "X_test_preprocessed.csv"))

y_train = pd.read_csv(os.path.join(PREPROCESSED_DIR, "y_train.csv")).squeeze("columns")
y_val = pd.read_csv(os.path.join(PREPROCESSED_DIR, "y_validation.csv")).squeeze("columns")
y_test = pd.read_csv(os.path.join(PREPROCESSED_DIR, "y_test.csv")).squeeze("columns")

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (18183, 81)
X_val: (3873, 81)
X_test: (3923, 81)

y_train: (18183,)
y_val: (3873,)
y_test: (3923,)


In [4]:
# Load preserved match IDs from preprocessing

train_ids = pd.read_csv(os.path.join(PREPROCESSED_DIR, "train_ids.csv"))["id"]
val_ids = pd.read_csv(os.path.join(PREPROCESSED_DIR, "validation_ids.csv"))["id"]
test_ids = pd.read_csv(os.path.join(PREPROCESSED_DIR, "test_ids.csv"))["id"]

print("Preserved match IDs loaded.")
print("Training IDs   :", len(train_ids))
print("Validation IDs :", len(val_ids))
print("Test IDs       :", len(test_ids))

assert train_ids.is_unique
assert val_ids.is_unique
assert test_ids.is_unique

assert len(train_ids) == len(X_train)
assert len(val_ids) == len(X_val)
assert len(test_ids) == len(X_test)

print("Match ID integrity verified.")

Preserved match IDs loaded.
Training IDs   : 18183
Validation IDs : 3873
Test IDs       : 3923
Match ID integrity verified.


Verify that raw and preprocessed rows still align

In [5]:
assert len(train_raw) == len(X_train), (
    "Training raw and preprocessed row counts do not match.")
assert len(val_raw) == len(X_val), (
    "Validation raw and preprocessed row counts do not match.")
assert len(test_raw) == len(X_test), (
    "Test raw and preprocessed row counts do not match.")

print("Row-count alignment verified.")

Row-count alignment verified.


In [6]:
# Verify preserved IDs correspond exactly to the raw splits

assert np.array_equal(train_ids.to_numpy(), train_raw["id"].to_numpy()
), "Training ID order mismatch."
assert np.array_equal(val_ids.to_numpy(), val_raw["id"].to_numpy()
), "Validation ID order mismatch."
assert np.array_equal(test_ids.to_numpy(), test_raw["id"].to_numpy()
), "Test ID order mismatch."

print("Raw/preprocessed match-ID alignment verified.")

Raw/preprocessed match-ID alignment verified.


Verify required historical columns

In [7]:
required_history_columns = [
    "id",
    "date",
    "home_team",
    "away_team",
    "home_team_goal",
    "away_team_goal",
    "target_outcome"
]

for name, d in [("train", train_raw), ("validation", val_raw), ("test", test_raw)]:
    missing_cols = [c for c in required_history_columns if c not in d.columns]
    if missing_cols:
        raise ValueError(f"{name}: required columns missing: {missing_cols}")

print("All required historical columns are available in train, validation and test.")

All required historical columns are available in train, validation and test.


In [8]:
# Post-match event columns (XML text recorded during/after the match) -> leakage
EVENT_COLUMNS = ["goal", "shoton", "shotoff", "foulcommit",
                 "card", "cross", "corner", "possession"]

# Player API IDs: identifiers, removed in Data Preprocessing
PLAYER_ID_COLUMNS = [f"{s}_player_{i}" for s in ("home", "away") for i in range(1, 12)]

# Raw bookmaker odds (the only columns allowed to keep genuine NaN values)
ODDS_COLS = [f"{b}{o}" for b in ("B365", "BW", "IW", "LB", "PS", "WH", "SJ", "VC", "GB", "BS")
             for o in ("H", "D", "A")]

LEAKAGE_COLUMNS = {"home_team_goal", "away_team_goal", "target_outcome",
                   "total_goals", "goal_difference"} | set(EVENT_COLUMNS)

print("Leakage columns monitored:", len(LEAKAGE_COLUMNS))

Leakage columns monitored: 13


Prepare raw historical event data

In [9]:
raw_frames = []

for df, split_name in [
    (train_raw, "train"),
    (val_raw, "validation"),
    (test_raw, "test")
]:

    temp = df.copy()

    temp["date"] = pd.to_datetime(
        temp["date"], errors="coerce"
    )

    temp["_split"] = split_name
    raw_frames.append(temp)

all_raw = pd.concat(
    raw_frames, ignore_index=True
)

all_raw = all_raw.sort_values(
    "date"
).reset_index(drop=True)

print("Combined historical dataset:")
print(all_raw.shape)

display(
    all_raw[
        [
            "date",
            "home_team",
            "away_team",
            "home_team_goal",
            "away_team_goal",
            "target_outcome",
            "_split"
        ]
    ].head(10)
)

Combined historical dataset:
(25979, 122)


,date,home_team,away_team,home_team_goal,away_team_goal,target_outcome,_split
0,2008-07-18,BSC Young Boys,FC Basel,1,2,Away Win,train
1,2008-07-19,FC Aarau,FC Sion,3,1,Home Win,train
2,2008-07-20,FC Luzern,FC Vaduz,1,2,Away Win,train
3,2008-07-20,Neuchâtel Xamax,FC Zürich,1,2,Away Win,train
4,2008-07-23,AC Bellinzona,Neuchâtel Xamax,1,2,Away Win,train
5,2008-07-23,FC Basel,Grasshopper Club Zürich,1,0,Home Win,train
6,2008-07-23,FC Zürich,FC Luzern,1,0,Home Win,train
7,2008-07-24,FC Sion,BSC Young Boys,2,1,Home Win,train
8,2008-07-24,FC Vaduz,FC Aarau,0,2,Away Win,train
9,2008-07-26,FC Luzern,Neuchâtel Xamax,0,1,Away Win,train


Check chronological ordering

In [10]:
if not all_raw["date"].is_monotonic_increasing:
    raise ValueError(
        "Historical dataset is not sorted chronologically."
    )

if all_raw["date"].isna().any():
    raise ValueError("Missing/invalid dates found in historical data.")

print("Historical data is chronologically ordered and has no missing dates.")

Historical data is chronologically ordered and has no missing dates.


Prepare target outcome labels

In [11]:
valid_outcomes = {
    "Home Win",
    "Draw",
    "Away Win"
}

invalid_outcomes = set(
    all_raw["target_outcome"].dropna().unique()
) - valid_outcomes

if invalid_outcomes:
    raise ValueError(
        f"Unexpected target values found: {invalid_outcomes}"
    )

print("Target outcome values verified.")

Target outcome values verified.


Historical team state

In [12]:
INITIAL_ELO = 1500.0

def create_team_state():
    return {
        # Overall historical performance
        "matches": 0,
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "points": 0,
        "goals_for": 0,
        "goals_against": 0,

        # Historical HOME performance
        "home_matches": 0,
        "home_wins": 0,
        "home_draws": 0,
        "home_losses": 0,
        "home_points": 0,
        "home_goals_for": 0,
        "home_goals_against": 0,

        # Historical AWAY performance
        "away_matches": 0,
        "away_wins": 0,
        "away_draws": 0,
        "away_losses": 0,
        "away_points": 0,
        "away_goals_for": 0,
        "away_goals_against": 0,

        # Rolling form - Keep last 10 so we can calculate 3, 5 and 10
        "recent_points": deque(maxlen=10),
        "recent_goals_for": deque(maxlen=10),
        "recent_goals_against": deque(maxlen=10),
        "recent_results": deque(maxlen=10),

        # Rest information
        "last_match_date": None,

        # Elo rating
        "elo": INITIAL_ELO
    }


team_history = defaultdict(create_team_state)

print("Enhanced historical team state initialized.")

Enhanced historical team state initialized.


Function to calculate historical features

In [13]:
MAX_REST_DAYS = 30.0     # anything longer is "fully rested"

def safe_rate(numerator, denominator):
    return numerator / denominator if denominator > 0 else 0.0

def rolling_average(values, window):
    values = list(values)[-window:]

    if len(values) == 0:
        return 0.0

    return float(np.mean(values))

def rolling_win_rate(results, window):
    results = list(results)[-window:]

    if len(results) == 0:
        return 0.0

    return sum(result == "W" for result in results) / len(results)

def calculate_team_features(
    history,
    team,
    match_date,
    venue
):
    state = history[team]

    matches = state["matches"]

    # Rest days
    if state["last_match_date"] is None:
        days_since_last_match = MAX_REST_DAYS            # unknown history -> neutral value
    else:
        days_since_last_match = (
            pd.Timestamp(match_date)
            - pd.Timestamp(state["last_match_date"])
        ).days

        # Defensive protection + cap off-season gaps
        days_since_last_match = min(
            max(float(days_since_last_match), 0.0),
            MAX_REST_DAYS
        )

    # Overall historical performance
    features = {
        "matches_before": matches,
        "win_rate": safe_rate(state["wins"], matches),
        "draw_rate": safe_rate(state["draws"], matches),
        "loss_rate": safe_rate(state["losses"], matches),
        "points_per_match": safe_rate(state["points"], matches),
        "goals_for_per_match": safe_rate(state["goals_for"], matches),
        "goals_against_per_match": safe_rate(state["goals_against"], matches),
        "goal_difference_per_match": safe_rate(
            state["goals_for"] - state["goals_against"], matches),

        # Rolling form: last 3
        "points_last3": rolling_average(state["recent_points"], 3),
        "goals_for_last3": rolling_average(state["recent_goals_for"], 3),
        "goals_against_last3": rolling_average(state["recent_goals_against"], 3),
        "win_rate_last3": rolling_win_rate(state["recent_results"], 3),

        # Rolling form: last 5
        "points_last5": rolling_average(state["recent_points"], 5),
        "goals_for_last5": rolling_average(state["recent_goals_for"], 5),
        "goals_against_last5": rolling_average(state["recent_goals_against"], 5),
        "win_rate_last5": rolling_win_rate(state["recent_results"], 5),

        # Rolling form: last 10
        "points_last10": rolling_average(state["recent_points"], 10),
        "goals_for_last10": rolling_average(state["recent_goals_for"], 10),
        "goals_against_last10": rolling_average(state["recent_goals_against"], 10),
        "win_rate_last10": rolling_win_rate(state["recent_results"], 10),

        # Rest + Elo
        "days_since_last_match": days_since_last_match,
        "elo_before": state["elo"]
    }

    # Venue-specific historical performance
    if venue == "home":

        venue_matches = state["home_matches"]

        features.update({
            "venue_matches_before":venue_matches,
            "venue_win_rate":
                safe_rate(state["home_wins"], venue_matches),
            "venue_draw_rate":
                safe_rate(state["home_draws"], venue_matches),
            "venue_points_per_match":
                safe_rate(state["home_points"], venue_matches),
            "venue_goals_for_per_match":
                safe_rate(state["home_goals_for"], venue_matches),
            "venue_goals_against_per_match":
                safe_rate(state["home_goals_against"], venue_matches)
        })

    elif venue == "away":

        venue_matches = state["away_matches"]

        features.update({
            "venue_matches_before":venue_matches,
            "venue_win_rate":
                safe_rate(state["away_wins"], venue_matches),
            "venue_draw_rate":
                safe_rate(state["away_draws"], venue_matches),
            "venue_points_per_match":
                safe_rate(state["away_points"], venue_matches),
            "venue_goals_for_per_match":
                safe_rate(state["away_goals_for"], venue_matches),
            "venue_goals_against_per_match":
                safe_rate(state["away_goals_against"], venue_matches)
        })

    else:
        raise ValueError(
            "venue must be either 'home' or 'away'"
        )

    return features

In [14]:
# Elo rating update
ELO_K = 20.0
HOME_ELO_ADVANTAGE = 100.0

def update_elo(history, home_team, away_team, result):
    home_elo = history[home_team]["elo"]
    away_elo = history[away_team]["elo"]

    # Expected home score including home advantage
    expected_home = 1 / (
        1 + 10 ** (
            (
                away_elo
                - (home_elo + HOME_ELO_ADVANTAGE)
            ) / 400
        )
    )

    expected_away = 1 - expected_home

    if result == "Home Win":
        actual_home = 1.0
        actual_away = 0.0

    elif result == "Draw":
        actual_home = 0.5
        actual_away = 0.5

    elif result == "Away Win":
        actual_home = 0.0
        actual_away = 1.0

    else:
        raise ValueError(
            f"Unexpected result: {result}"
        )

    history[home_team]["elo"] = (
        home_elo
        + ELO_K * (
            actual_home - expected_home
        )
    )

    history[away_team]["elo"] = (
        away_elo
        + ELO_K * (
            actual_away - expected_away
        )
    )

Function to update historical team information

In [15]:
def update_team_history(
    history,
    home_team,
    away_team,
    home_goals,
    away_goals,
    result,
    match_date
):
    home_state = history[home_team]
    away_state = history[away_team]

    # Overall match counts
    home_state["matches"] += 1
    away_state["matches"] += 1

    # Overall goals
    home_state["goals_for"] += home_goals
    home_state["goals_against"] += away_goals
    away_state["goals_for"] += away_goals
    away_state["goals_against"] += home_goals

    # Venue-specific counts and goals
    home_state["home_matches"] += 1
    away_state["away_matches"] += 1
    home_state["home_goals_for"] += home_goals
    home_state["home_goals_against"] += away_goals
    away_state["away_goals_for"] += away_goals
    away_state["away_goals_against"] += home_goals

    # Result updates
    if result == "Home Win":

        # Overall
        home_state["wins"] += 1
        home_state["points"] += 3
        away_state["losses"] += 1

        # Venue specific
        home_state["home_wins"] += 1
        home_state["home_points"] += 3
        away_state["away_losses"] += 1

        # Recent form
        home_state["recent_points"].append(3)
        away_state["recent_points"].append(0)
        home_state["recent_results"].append("W")
        away_state["recent_results"].append("L")

    elif result == "Draw":

        # Overall
        home_state["draws"] += 1
        away_state["draws"] += 1
        home_state["points"] += 1
        away_state["points"] += 1

        # Venue specific
        home_state["home_draws"] += 1
        away_state["away_draws"] += 1
        home_state["home_points"] += 1
        away_state["away_points"] += 1

        # Recent form
        home_state["recent_points"].append(1)
        away_state["recent_points"].append(1)
        home_state["recent_results"].append("D")
        away_state["recent_results"].append("D")

    elif result == "Away Win":

        # Overall
        home_state["losses"] += 1
        away_state["wins"] += 1
        away_state["points"] += 3

        # Venue specific
        home_state["home_losses"] += 1
        away_state["away_wins"] += 1
        away_state["away_points"] += 3

        # Recent form
        home_state["recent_points"].append(0)
        away_state["recent_points"].append(3)
        home_state["recent_results"].append("L")
        away_state["recent_results"].append("W")

    else:
        raise ValueError(f"Unexpected match result: {result}")

    # Recent goals
    home_state["recent_goals_for"].append(
        home_goals
    )
    home_state["recent_goals_against"].append(
        away_goals
    )
    away_state["recent_goals_for"].append(
        away_goals
    )
    away_state["recent_goals_against"].append(
        home_goals
    )

    # Elo update
    update_elo(
        history=history,
        home_team=home_team,
        away_team=away_team,
        result=result
    )

    # Last match date
    home_state["last_match_date"] = match_date
    away_state["last_match_date"] = match_date

Generate features without using same-day results

In [16]:
engineered_rows = []

team_history = defaultdict(create_team_state)

for match_date, daily_matches in all_raw.groupby(
    "date",
    sort=True
):

    # STEP 1: Generate features using ONLY information available before this date.

    for _, row in daily_matches.iterrows():

        home_team = row["home_team"]
        away_team = row["away_team"]

        home_features = calculate_team_features(
            history=team_history,
            team=home_team,
            match_date=match_date,
            venue="home"
        )

        away_features = calculate_team_features(
            history=team_history,
            team=away_team,
            match_date=match_date,
            venue="away"
        )

        engineered_rows.append({

            # Match identity
            "_match_id": row["id"],
            "date": match_date,
            "home_team": home_team,
            "away_team": away_team,

            # HOME: overall historical performance
            "home_matches_before":home_features["matches_before"],
            "home_win_rate":home_features["win_rate"],
            "home_draw_rate":home_features["draw_rate"],
            "home_loss_rate":home_features["loss_rate"],
            "home_points_per_match":home_features["points_per_match"],
            "home_goals_for_per_match":home_features["goals_for_per_match"],
            "home_goals_against_per_match":home_features["goals_against_per_match"],
            "home_goal_difference_per_match":home_features["goal_difference_per_match"],

            # HOME: performance specifically at home
            "home_home_matches_before":home_features["venue_matches_before"],
            "home_home_win_rate":home_features["venue_win_rate"],
            "home_home_draw_rate":home_features["venue_draw_rate"],
            "home_home_points_per_match":home_features["venue_points_per_match"],
            "home_home_goals_for_per_match":home_features["venue_goals_for_per_match"],
            "home_home_goals_against_per_match":home_features["venue_goals_against_per_match"],

            # HOME: rolling form
            "home_points_last3":home_features["points_last3"],
            "home_points_last5":home_features["points_last5"],
            "home_points_last10":home_features["points_last10"],
            "home_goals_for_last3":home_features["goals_for_last3"],
            "home_goals_for_last5":home_features["goals_for_last5"],
            "home_goals_for_last10":home_features["goals_for_last10"],
            "home_goals_against_last3":home_features["goals_against_last3"],
            "home_goals_against_last5":home_features["goals_against_last5"],
            "home_goals_against_last10":home_features["goals_against_last10"],
            "home_win_rate_last3":home_features["win_rate_last3"],
            "home_win_rate_last5":home_features["win_rate_last5"],
            "home_win_rate_last10":home_features["win_rate_last10"],

            # HOME: rest and Elo
            "home_days_since_last_match":home_features["days_since_last_match"],
            "home_elo_before":home_features["elo_before"],

            # AWAY: overall historical performance
            "away_matches_before":away_features["matches_before"],
            "away_win_rate":away_features["win_rate"],
            "away_draw_rate":away_features["draw_rate"],
            "away_loss_rate":away_features["loss_rate"],
            "away_points_per_match":away_features["points_per_match"],
            "away_goals_for_per_match":away_features["goals_for_per_match"],
            "away_goals_against_per_match":away_features["goals_against_per_match"],
            "away_goal_difference_per_match":away_features["goal_difference_per_match"],

            # AWAY: performance specifically away
            "away_away_matches_before":away_features[ "venue_matches_before"],
            "away_away_win_rate":away_features["venue_win_rate"],
            "away_away_draw_rate":away_features["venue_draw_rate"],
            "away_away_points_per_match":away_features["venue_points_per_match"],
            "away_away_goals_for_per_match":away_features["venue_goals_for_per_match"],

            "away_away_goals_against_per_match":away_features["venue_goals_against_per_match"],

            # AWAY: rolling form
            "away_points_last3":away_features["points_last3"],
            "away_points_last5":away_features["points_last5"],
            "away_points_last10":away_features["points_last10"],
            "away_goals_for_last3":away_features["goals_for_last3"],
            "away_goals_for_last5":away_features["goals_for_last5"],
            "away_goals_for_last10":away_features["goals_for_last10"],
            "away_goals_against_last3":away_features["goals_against_last3"],
            "away_goals_against_last5":away_features["goals_against_last5"],
            "away_goals_against_last10":away_features["goals_against_last10"],
            "away_win_rate_last3":away_features["win_rate_last3"],
            "away_win_rate_last5":away_features["win_rate_last5"],
            "away_win_rate_last10":away_features["win_rate_last10"],

            # AWAY: rest and Elo
            "away_days_since_last_match":away_features["days_since_last_match"],
            "away_elo_before":away_features["elo_before"]
        })

    # STEP 2: Only AFTER all features for the date are generated,
    # update team histories using that day's results.

    for _, row in daily_matches.iterrows():

        update_team_history(
            history=team_history,
            home_team=row["home_team"],
            away_team=row["away_team"],
            home_goals=row["home_team_goal"],
            away_goals=row["away_team_goal"],
            result=row["target_outcome"],
            match_date=match_date
        )

print(
    "Historical features generated:",
    len(engineered_rows)
)

Historical features generated: 25979


Create engineered feature DataFrame

In [17]:
engineered_df = pd.DataFrame(
    engineered_rows
)

print("Engineered dataset shape:")
print(engineered_df.shape)

display(engineered_df.head(10))

Engineered dataset shape:
(25979, 60)


,_match_id,date,home_team,away_team,home_matches_before,home_win_rate,home_draw_rate,home_loss_rate,home_points_per_match,home_goals_for_per_match,...,away_goals_for_last5,away_goals_for_last10,away_goals_against_last3,away_goals_against_last5,away_goals_against_last10,away_win_rate_last3,away_win_rate_last5,away_win_rate_last10,away_days_since_last_match,away_elo_before
0,24559,2008-07-18,BSC Young Boys,FC Basel,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,1500.000000
1,24560,2008-07-19,FC Aarau,FC Sion,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,1500.000000
2,24561,2008-07-20,FC Luzern,FC Vaduz,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,1500.000000
3,24562,2008-07-20,Neuchâtel Xamax,FC Zürich,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,1500.000000
4,24614,2008-07-23,AC Bellinzona,Neuchâtel Xamax,0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,2.0,2.0,2.0,0.0,0.0,0.0,3.0,1487.198700
5,24613,2008-07-23,FC Basel,Grasshopper Club Zürich,1,1.0,0.0,0.0,3.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,1500.000000
6,24615,2008-07-23,FC Zürich,FC Luzern,1,1.0,0.0,0.0,3.0,2.0,...,1.0,1.0,2.0,2.0,2.0,0.0,0.0,0.0,3.0,1487.198700
7,24616,2008-07-24,FC Sion,BSC Young Boys,1,0.0,0.0,1.0,0.0,1.0,...,1.0,1.0,2.0,2.0,2.0,0.0,0.0,0.0,6.0,1487.198700
8,24617,2008-07-24,FC Vaduz,FC Aarau,1,1.0,0.0,0.0,3.0,2.0,...,3.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,5.0,1507.198700
9,24669,2008-07-26,FC Luzern,Neuchâtel Xamax,2,0.0,0.0,1.0,0.0,0.5,...,1.5,1.5,1.5,1.5,1.5,0.5,0.5,0.5,3.0,1500.335918


Verify the first match has no artificial history

In [18]:
first_match = engineered_df.iloc[0]

print("Home matches before:", first_match["home_matches_before"])
print("Away matches before:",first_match["away_matches_before"])

Home matches before: 0
Away matches before: 0


In [19]:
print("Home Elo before:", first_match["home_elo_before"])
print("Away Elo before:", first_match["away_elo_before"])

assert first_match["home_elo_before"] == INITIAL_ELO
assert first_match["away_elo_before"] == INITIAL_ELO

print("Initial-history check passed.")

Home Elo before: 1500.0
Away Elo before: 1500.0
Initial-history check passed.


In [20]:
no_history = engineered_df[
    (engineered_df["home_matches_before"] == 0) |
    (engineered_df["away_matches_before"] == 0)
]

print("Matches involving a team with no prior history:", len(no_history))

Matches involving a team with no prior history: 204


Create relative team-strength features

In [21]:
# Overall historical differences
engineered_df["win_rate_difference"] = (
    engineered_df["home_win_rate"] - engineered_df["away_win_rate"]
)

engineered_df["draw_rate_difference"] = (
    engineered_df["home_draw_rate"] - engineered_df["away_draw_rate"]
)

engineered_df["points_per_match_difference"] = (
    engineered_df["home_points_per_match"] - engineered_df["away_points_per_match"]
)

engineered_df["goals_for_difference"] = (
    engineered_df["home_goals_for_per_match"] - engineered_df["away_goals_for_per_match"]
)

engineered_df["goals_against_difference"] = (
    engineered_df["home_goals_against_per_match"] - engineered_df["away_goals_against_per_match"]
)

# Venue-specific strength differences
engineered_df["venue_win_rate_difference"] = (
    engineered_df["home_home_win_rate"] - engineered_df["away_away_win_rate"]
)

engineered_df["venue_points_difference"] = (
    engineered_df["home_home_points_per_match"] - engineered_df["away_away_points_per_match"]
)

engineered_df["venue_goals_for_difference"] = (
    engineered_df["home_home_goals_for_per_match"] - engineered_df["away_away_goals_for_per_match"]
)

engineered_df["venue_goals_against_difference"] = (
    engineered_df["home_home_goals_against_per_match"] - engineered_df["away_away_goals_against_per_match"]
)

# Rolling form differences
for window in [3, 5, 10]:

    engineered_df[f"points_last{window}_difference"
    ] = (
        engineered_df[f"home_points_last{window}"
        ]
        - engineered_df[f"away_points_last{window}"
        ]
    )

    engineered_df[f"goals_for_last{window}_difference"
    ] = (
        engineered_df[f"home_goals_for_last{window}"
        ]
        - engineered_df[f"away_goals_for_last{window}"
        ]
    )

    engineered_df[f"goals_against_last{window}_difference"
    ] = (
        engineered_df[f"home_goals_against_last{window}"
        ]
        - engineered_df[f"away_goals_against_last{window}"
        ]
    )

    engineered_df[f"win_rate_last{window}_difference"
    ] = (
        engineered_df[f"home_win_rate_last{window}"
        ]
        - engineered_df[f"away_win_rate_last{window}"
        ]
    )

# Rest difference
engineered_df["rest_days_difference"] = (
    engineered_df["home_days_since_last_match"]
    - engineered_df["away_days_since_last_match"]
)

# Elo difference
engineered_df["elo_difference"] = (
    engineered_df["home_elo_before"]
    - engineered_df["away_elo_before"]
)

print("Relative team-strength features created.")

Relative team-strength features created.


Inspect all engineered features

In [22]:
engineered_columns = [
    col
    for col in engineered_df.columns
    if col not in [
        "_match_id",
        "date",
        "home_team",
        "away_team"
    ]
]

print("Number of engineered features:", len(engineered_columns))

for col in engineered_columns:
    print("-", col)

Number of engineered features: 79
- home_matches_before
- home_win_rate
- home_draw_rate
- home_loss_rate
- home_points_per_match
- home_goals_for_per_match
- home_goals_against_per_match
- home_goal_difference_per_match
- home_home_matches_before
- home_home_win_rate
- home_home_draw_rate
- home_home_points_per_match
- home_home_goals_for_per_match
- home_home_goals_against_per_match
- home_points_last3
- home_points_last5
- home_points_last10
- home_goals_for_last3
- home_goals_for_last5
- home_goals_for_last10
- home_goals_against_last3
- home_goals_against_last5
- home_goals_against_last10
- home_win_rate_last3
- home_win_rate_last5
- home_win_rate_last10
- home_days_since_last_match
- home_elo_before
- away_matches_before
- away_win_rate
- away_draw_rate
- away_loss_rate
- away_points_per_match
- away_goals_for_per_match
- away_goals_against_per_match
- away_goal_difference_per_match
- away_away_matches_before
- away_away_win_rate
- away_away_draw_rate
- away_away_points_per_match

Verify no leakage columns are in engineered features

In [23]:
# 1. Direct target / post-match leakage check (LEAKAGE_COLUMNS includes the event columns)
leakage_in_features = set(engineered_columns) & LEAKAGE_COLUMNS
if leakage_in_features:
    raise ValueError(
        f"Leakage detected in engineered features: {leakage_in_features}"
    )
print("No direct target-leakage columns found.")

# 2. Historical feature integrity checks
for col in ["home_matches_before", "away_matches_before"]:
    if not engineered_df[col].ge(0).all():
        raise ValueError(f"Invalid historical feature: {col} contains negative values.")
print("Historical match-count features are valid.")

# 3. NaN check + cold-start disclosure
missing_historical = engineered_df[engineered_columns].isna().sum()
missing_historical = missing_historical[missing_historical > 0]
if not missing_historical.empty:
    raise ValueError(
        "Missing values detected in engineered features:\n"
        f"{missing_historical}"
    )

cold = (engineered_df["home_matches_before"] == 0) | (engineered_df["away_matches_before"] == 0)
print(f"No NaN values – BUT {cold.sum()} cold-start rows contain 0.0 placeholders "
      f"(no real history). They must be filtered with a burn-in rule (see Final_Preprocessing).")

# 4. Infinite values
numeric_engineered = engineered_df[engineered_columns].select_dtypes(include=np.number)
if np.isinf(numeric_engineered.to_numpy()).any():
    raise ValueError("Infinite values detected in engineered features.")
print("No infinite values found in engineered features.")

# 5. Real temporal-leakage audit: matches_before must equal the number of STRICTLY earlier matches
long_hist = pd.concat([
    all_raw[["date", "home_team"]].rename(columns={"home_team": "team"}),
    all_raw[["date", "away_team"]].rename(columns={"away_team": "team"}),
])
for _, r in engineered_df.sample(500, random_state=42).iterrows():
    exp_home = ((long_hist["team"] == r["home_team"]) & (long_hist["date"] < r["date"])).sum()
    exp_away = ((long_hist["team"] == r["away_team"]) & (long_hist["date"] < r["date"])).sum()
    assert r["home_matches_before"] == exp_home, f"Future info leak (home) in match {r['_match_id']}"
    assert r["away_matches_before"] == exp_away, f"Future info leak (away) in match {r['_match_id']}"
print("Temporal audit passed: features use only strictly earlier matches.")

print("\nLeakage and historical feature integrity checks passed.")

No direct target-leakage columns found.
Historical match-count features are valid.
No NaN values – BUT 204 cold-start rows contain 0.0 placeholders (no real history). They must be filtered with a burn-in rule (see Final_Preprocessing).
No infinite values found in engineered features.
Temporal audit passed: features use only strictly earlier matches.

Leakage and historical feature integrity checks passed.


Verify engineered feature quality

In [24]:
feature_quality = pd.DataFrame({
    "Feature": engineered_columns,
    "Missing Values": [
        engineered_df[col].isnull().sum()
        for col in engineered_columns
    ],
    "Unique Values": [
        engineered_df[col].nunique()
        for col in engineered_columns
    ]
})

display(feature_quality)

,Feature,Missing Values,Unique Values
0,home_matches_before,0,304
1,home_win_rate,0,6762
2,home_draw_rate,0,4121
3,home_loss_rate,0,6441
4,home_points_per_match,0,10578
...,...,...,...
74,goals_for_last10_difference,0,605
75,goals_against_last10_difference,0,557
76,win_rate_last10_difference,0,252
77,rest_days_difference,0,55


Restore the original split

In [25]:
split_lookup = all_raw[
    ["id", "_split"]
].copy()

split_lookup = split_lookup.rename(
    columns={"id": "_match_id"}
)

engineered_df = engineered_df.merge(
    split_lookup,
    on="_match_id",
    how="left",
    validate="one_to_one"
)

print(engineered_df["_split"].value_counts())

_split
train         18183
test           3923
validation     3873
Name: count, dtype: int64


Verify split-specific feature counts

In [26]:
print("Feature rows by split:")

display(
    engineered_df.groupby("_split")
    .agg(
        Rows=("_match_id", "count"),
        Start_Date=("date", "min"),
        End_Date=("date", "max")
    )
)

Feature rows by split:


,Rows,Start_Date,End_Date
_split,,,
test,3923,2015-04-11,2016-05-25
train,18183,2008-07-18,2014-02-10
validation,3873,2014-02-11,2015-04-10


Create engineered feature matrices

In [27]:
# Reset row indexes
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

In [28]:
# Attach preserved match IDs to preprocessed feature matrices

X_train["_match_id"] = train_ids.reset_index(drop=True)
X_val["_match_id"] = val_ids.reset_index(drop=True)
X_test["_match_id"] = test_ids.reset_index(drop=True)

assert X_train["_match_id"].is_unique
assert X_val["_match_id"].is_unique
assert X_test["_match_id"].is_unique

print("Preserved match IDs attached to feature matrices.")

Preserved match IDs attached to feature matrices.


In [29]:
# Create the engineered feature subsets
train_engineered = (
    engineered_df[ engineered_df["_split"] == "train"]
    .sort_values("date")
    .reset_index(drop=True)
)

val_engineered = (
    engineered_df[ engineered_df["_split"] == "validation"]
    .sort_values("date")
    .reset_index(drop=True)
)

test_engineered = (
    engineered_df[ engineered_df["_split"] == "test"]
    .sort_values("date")
    .reset_index(drop=True)
)

In [30]:
assert train_engineered["_match_id"].is_unique
assert val_engineered["_match_id"].is_unique
assert test_engineered["_match_id"].is_unique

print("Engineered match IDs verified as unique.")

Engineered match IDs verified as unique.


Verify row alignment

In [31]:
assert len(X_train) == len(train_engineered)
assert len(X_val) == len(val_engineered)
assert len(X_test) == len(test_engineered)

print("Row alignment verified.")

Row alignment verified.


Verify dates and teams align

In [32]:
# Verify exact match-ID alignment
def verify_id_alignment(
    X,
    engineered,
    name
):
    x_ids = set(X["_match_id"])

    engineered_ids = set(engineered["_match_id"])

    if x_ids != engineered_ids:

        missing_from_engineered =  x_ids - engineered_ids
        missing_from_X =  engineered_ids - x_ids

        raise ValueError(
            f"{name} ID alignment failed.\n"
            f"Missing from engineered: "
            f"{len(missing_from_engineered)}\n"
            f"Missing from X: "
            f"{len(missing_from_X)}"
        )

    print(f"{name}: exact match-ID alignment verified.")

verify_id_alignment(
    X_train,
    train_engineered,
    "Training"
)

verify_id_alignment(
    X_val,
    val_engineered,
    "Validation"
)

verify_id_alignment(
    X_test,
    test_engineered,
    "Test"
)


# Verify dates and teams (not only IDs) line up row-for-row by match ID
def verify_content_alignment(X, engineered, name):
    m = X[["_match_id", "date", "home_team", "away_team"]].merge(
        engineered[["_match_id", "date", "home_team", "away_team"]],
        on="_match_id", suffixes=("_x", "_eng"), validate="one_to_one")
    m["date_x"] = pd.to_datetime(m["date_x"])
    assert (m["date_x"]      == m["date_eng"]).all(),      f"{name}: date mismatch"
    assert (m["home_team_x"] == m["home_team_eng"]).all(), f"{name}: home team mismatch"
    assert (m["away_team_x"] == m["away_team_eng"]).all(), f"{name}: away team mismatch"
    print(f"{name}: date/team alignment verified.")

verify_content_alignment(X_train, train_engineered, "Training")
verify_content_alignment(X_val,   val_engineered,   "Validation")
verify_content_alignment(X_test,  test_engineered,  "Test")

Training: exact match-ID alignment verified.
Validation: exact match-ID alignment verified.
Test: exact match-ID alignment verified.
Training: date/team alignment verified.
Validation: date/team alignment verified.
Test: date/team alignment verified.


Select only engineered predictors

In [33]:
ENGINEERED_FEATURE_COLUMNS = [
    col
    for col in engineered_df.columns
    if col not in [
        "_match_id",
        "date",
        "home_team",
        "away_team",
        "_split"
    ]
]

print(
    "Total engineered predictors:",
    len(ENGINEERED_FEATURE_COLUMNS)
)

Total engineered predictors: 79


Combine Preprocessed Features + Engineered Features

In [34]:
def merge_engineered_features(
    X,
    engineered
):
    X = X.copy()

    # IDs must already exist and be unique
    if "_match_id" not in X.columns:
        raise ValueError("_match_id is missing from preprocessed features.")

    if not X["_match_id"].is_unique:
        raise ValueError("Duplicate match IDs found in X.")

    if not engineered["_match_id"].is_unique:
        raise ValueError("Duplicate match IDs found in engineered data.")

    # Merge explicitly by immutable match ID
    X_final = X.merge(
        engineered[["_match_id"] + ENGINEERED_FEATURE_COLUMNS],
        on="_match_id",
        how="left",
        validate="one_to_one"
    )

    # Ensure every match received engineered features
    engineered_missing = (
        X_final[ENGINEERED_FEATURE_COLUMNS]
        .isnull()
        .any(axis=1)
        .sum()
    )

    if engineered_missing > 0:
        raise ValueError(
            f"{engineered_missing} rows failed to receive engineered features."
        )

    return X_final

In [35]:
X_train_final = merge_engineered_features(
    X_train,
    train_engineered
)

X_val_final = merge_engineered_features(
    X_val,
    val_engineered
)

X_test_final = merge_engineered_features(
    X_test,
    test_engineered
)

Remove raw date from final modelling data

In [36]:
# Drop identifiers/dates, plus event columns as a second line of defence
for df in [X_train_final, X_val_final, X_test_final]:
    df.drop(columns=["_match_id", "date"] + EVENT_COLUMNS, errors="ignore", inplace=True)

In [37]:
# Match identifiers must never reach the model

for name, df in [
    ("Training", X_train_final),
    ("Validation", X_val_final),
    ("Test", X_test_final)
]:
    assert "_match_id" not in df.columns
    assert "id" not in df.columns
    assert "match_id" not in df.columns
    assert not any(c in df.columns for c in PLAYER_ID_COLUMNS), \
        "Player ID columns found - re-run the corrected Data Preprocessing notebook"

    print(f"{name}: identifier removal verified.")

Training: identifier removal verified.
Validation: identifier removal verified.
Test: identifier removal verified.


Final feature matrix check

In [38]:
print("Final feature matrix shapes")
print("-" * 40)

print("Training   :", X_train_final.shape)
print("Validation :", X_val_final.shape)
print("Test       :", X_test_final.shape)

Final feature matrix shapes
----------------------------------------
Training   : (18183, 159)
Validation : (3873, 159)
Test       : (3923, 159)


Check final columns for leakage

In [39]:
final_leakage_columns = [
    col
    for col in (
        set(X_train_final.columns)
        & LEAKAGE_COLUMNS
    )
]

print("Potential leakage columns:")
print(final_leakage_columns)

if final_leakage_columns:
    raise ValueError(
        "Leakage columns remain in final feature matrix."
    )

print("\nFinal leakage check passed.")

Potential leakage columns:
[]

Final leakage check passed.


Check missing and infinite values

In [40]:
for name, d in [
    ("Training", X_train_final),
    ("Validation", X_val_final),
    ("Test", X_test_final)]:

    non_odds_missing = d.drop(columns=ODDS_COLS, errors="ignore").isnull().sum().sum()

    print(f"{name}: total missing = {d.isnull().sum().sum()}, outside odds columns = {non_odds_missing}")
    assert non_odds_missing == 0

Training: total missing = 115851, outside odds columns = 0
Validation: total missing = 41700, outside odds columns = 0
Test: total missing = 45909, outside odds columns = 0


In [41]:
for name, df in [
    ("Training", X_train_final),
    ("Validation", X_val_final),
    ("Test", X_test_final)
]:

    numeric_part = df.select_dtypes(
        include=np.number
    )

    print(
        name,
        "infinite values:",
        np.isinf(numeric_part).sum().sum()
    )

Training infinite values: 0
Validation infinite values: 0
Test infinite values: 0


Feature summary

In [42]:
feature_summary = pd.DataFrame({
    "Feature": X_train_final.columns,
    "Data Type": X_train_final.dtypes.astype(str),
    "Unique Values": [
        X_train_final[col].nunique()
        for col in X_train_final.columns
    ],
    "Missing Values": [
        X_train_final[col].isnull().sum()
        for col in X_train_final.columns
    ]
})

display(feature_summary)

,Feature,Data Type,Unique Values,Missing Values
season,season,object,6,0
stage,stage,int64,38,0
home_player_X1,home_player_X1,float64,3,0
home_player_X2,home_player_X2,float64,8,0
home_player_X3,home_player_X3,float64,7,0
...,...,...,...,...
goals_for_last10_difference,goals_for_last10_difference,float64,560,0
goals_against_last10_difference,goals_against_last10_difference,float64,510,0
win_rate_last10_difference,win_rate_last10_difference,float64,243,0
rest_days_difference,rest_days_difference,float64,54,0


Save final feature-engineered datasets

In [43]:
X_train_final.to_csv(
    os.path.join(FEATURE_DIR, "X_train_features.csv"),
    index=False
)

X_val_final.to_csv(
    os.path.join(FEATURE_DIR, "X_validation_features.csv"),
    index=False
)

X_test_final.to_csv(
    os.path.join(FEATURE_DIR, "X_test_features.csv"),
    index=False
)

y_train.to_csv(
    os.path.join(FEATURE_DIR, "y_train_features.csv"),
    index=False
)

y_val.to_csv(
    os.path.join(FEATURE_DIR, "y_validation_features.csv"),
    index=False
)

y_test.to_csv(
    os.path.join(FEATURE_DIR, "y_test_features.csv"),
    index=False
)

print("Feature-engineered datasets saved.")
print(os.listdir(FEATURE_DIR))

Feature-engineered datasets saved.
['X_train_features.csv', 'X_validation_features.csv', 'X_test_features.csv', 'y_train_features.csv', 'y_validation_features.csv', 'y_test_features.csv']


Save the feature dictionary

In [44]:
METRICS = {
    "matches_before": "number of matches played before this match",
    "win_rate": "share of previous matches won",
    "draw_rate": "share of previous matches drawn",
    "loss_rate": "share of previous matches lost",
    "points_per_match": "average league points per previous match",
    "goals_for_per_match": "average goals scored per previous match",
    "goals_against_per_match": "average goals conceded per previous match",
    "goal_difference_per_match": "average goal difference per previous match",
    "days_since_last_match": "days of rest since last match (capped at 30)",
    "elo_before": "Elo rating before the match",
}

# Names used by the "*_difference" features
DIFF_METRICS = {
    "points": METRICS["points_per_match"],
    "goals_for": METRICS["goals_for_per_match"],
    "goals_against": METRICS["goals_against_per_match"],
    "rest_days": METRICS["days_since_last_match"],
    "elo": METRICS["elo_before"],
}

def describe_metric(base):
    if base in METRICS:
        return METRICS[base]
    for w in (3, 5, 10):
        tag = f"_last{w}"
        if base.endswith(tag):
            core = base[:-len(tag)]
            if core == "win_rate":
                return f"win rate over the last {w} matches"
            return f"average {core.replace('_', ' ')} over the last {w} matches"
    return base.replace("_", " ")

def describe_feature(f):
    if f.endswith("_difference"):
        core = f[:-len("_difference")]
        if core.startswith("venue_"):
            core = core[len("venue_"):]
            who = "Home team's home record minus away team's away record"
        else:
            who = "Home team minus away team"
        return f"{who} – {DIFF_METRICS.get(core) or describe_metric(core)}"
    if f.startswith("home_home_"):   who, base = "Home team (home matches only)", f[10:]
    elif f.startswith("away_away_"): who, base = "Away team (away matches only)", f[10:]
    elif f.startswith("home_"):      who, base = "Home team", f[5:]
    elif f.startswith("away_"):      who, base = "Away team", f[5:]
    else:                            who, base = "Match", f
    return f"{who} – {describe_metric(base)}"

feature_dictionary = pd.DataFrame({"Feature": ENGINEERED_FEATURE_COLUMNS})
feature_dictionary["Description"] = feature_dictionary["Feature"].apply(describe_feature)

print("Feature dictionary entries:", len(feature_dictionary))

Feature dictionary entries: 79


In [45]:
assert len(feature_dictionary) == len(ENGINEERED_FEATURE_COLUMNS)

display(feature_dictionary)

feature_dictionary.to_csv(
    os.path.join(FEATURE_DIR, "feature_dictionary.csv"),
    index=False
)

print("Feature dictionary saved.")

,Feature,Description
0,home_matches_before,Home team – number of matches played before th...
1,home_win_rate,Home team – share of previous matches won
2,home_draw_rate,Home team – share of previous matches drawn
3,home_loss_rate,Home team – share of previous matches lost
4,home_points_per_match,Home team – average league points per previous...
...,...,...
74,goals_for_last10_difference,Home team minus away team – average goals for ...
75,goals_against_last10_difference,Home team minus away team – average goals agai...
76,win_rate_last10_difference,Home team minus away team – win rate over the ...
77,rest_days_difference,Home team minus away team – days of rest since...


Feature dictionary saved.
